# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Utsabsinha19/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

I will use a Random Forest model as the first ML method for the lane. It can learn non-linear relationships between multiple content and search-performance signals without requiring a complex model architecture. It also provides feature-importance information that can help explain which available signals the model relies on. The model will be evaluated against the simple baseline using the same data split and Precision@50 metric.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.ensemble import RandomForestClassifier

print("Model: Random Forest")
print("Reason: captures non-linear relationships and supports feature-importance analysis.")
print("Evaluation metric: Precision@50")

Model: Random Forest
Reason: captures non-linear relationships and supports feature-importance analysis.
Evaluation metric: Precision@50


## 2. Split design

I will use a client-grouped split so that a client's pages are not mixed between training and test sets. This is more honest for the intended use because the model should be evaluated on clients it did not directly learn from. I will use the same split when comparing the model with the baseline.

In [3]:
import pandas as pd
import numpy as np

url = "https://raw.githubusercontent.com/flyrank-bih/flyrank-ml-internship-starter/main/data/raw/content_refresh_anonymized.csv"

df = pd.read_csv(url)

print("Dataset loaded successfully")
print("Rows:", len(df))
print("Columns:", len(df.columns))

Dataset loaded successfully
Rows: 30000
Columns: 44


In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import GroupShuffleSplit

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(df, groups=df["client_id"])
)

train_df = df.iloc[train_idx].copy()
test_df = df.iloc[test_idx].copy()

print("Training rows:", len(train_df))
print("Test rows:", len(test_df))
print("Training clients:", train_df["client_id"].nunique())
print("Test clients:", test_df["client_id"].nunique())

print(
    "Client overlap:",
    len(
        set(train_df["client_id"])
        & set(test_df["client_id"])
    )
)

Training rows: 23837
Test rows: 6163
Training clients: 25
Test clients: 7
Client overlap: 0


## 3. Train + compare vs my baseline

I will create a provisional binary review-priority proxy from observed historical signals and train the model to rank higher-priority pages above lower-priority pages. The proxy is only a decision-support label and does not represent a causal outcome. The baseline and model will both be evaluated on the held-out test clients using Precision@50

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score

# -------------------------------------------------------
# Create a provisional review-priority proxy
# -------------------------------------------------------

work_df = df.copy()

# Recent impression change
work_df["impression_change_pct"] = np.where(
    work_df["impressions_prev_30d"] > 0,
    (
        work_df["impressions_last_30d"]
        - work_df["impressions_prev_30d"]
    ) / work_df["impressions_prev_30d"],
    0
)

# Define proxy from observed signals.
# High exposure + weak position + recent decline.
work_df["review_proxy_score"] = (
    0.40 * work_df["impressions_90d"].rank(pct=True)
    + 0.35 * work_df["avg_position"].rank(pct=True)
    + 0.25 * (-work_df["impression_change_pct"]).rank(pct=True)
)

# Top 25% = provisional positive class
threshold = work_df["review_proxy_score"].quantile(0.75)

work_df["review_label"] = (
    work_df["review_proxy_score"] >= threshold
).astype(int)

print("Positive-label rate:",
      round(work_df["review_label"].mean(), 4))

Positive-label rate: 0.25


In [6]:
feature_columns = [
    "search_volume",
    "competition",
    "cpc",
    "word_count",
    "char_count",
    "impressions_90d",
    "clicks_90d",
    "pageviews_90d",
    "sessions_90d",
    "users_90d",
    "engaged_sessions_90d",
    "ai_sessions_90d",
    "scroll_events_90d",
    "days_with_impressions",
    "days_with_sessions",
    "impressions_last_30d",
    "clicks_last_30d",
    "sessions_last_30d",
    "impressions_prev_30d",
    "clicks_prev_30d",
    "sessions_prev_30d",
    "content_age_days",
    "age_tier_order",
    "days_since_last_update"
]

X = work_df[feature_columns].copy()

X = X.fillna(X.median(numeric_only=True))

y = work_df["review_label"]

groups = work_df["client_id"]

gss = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42
)

train_idx, test_idx = next(
    gss.split(X, y, groups=groups)
)

X_train = X.iloc[train_idx]
X_test = X.iloc[test_idx]

y_train = y.iloc[train_idx]
y_test = y.iloc[test_idx]

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

X_train: (23837, 24)
X_test: (6163, 24)


In [7]:
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    min_samples_leaf=20,
    random_state=42,
    n_jobs=-1
)

model.fit(X_train, y_train)

model_scores = model.predict_proba(X_test)[:, 1]

print("Model trained successfully.")

Model trained successfully.


In [8]:
def precision_at_k(y_true, scores, k=50):
    order = np.argsort(scores)[::-1][:k]
    return y_true.iloc[order].mean()

model_p50 = precision_at_k(
    y_test.reset_index(drop=True),
    model_scores,
    50
)

print("Model Precision@50:", round(model_p50, 4))

Model Precision@50: 0.72


In [9]:
baseline_test = work_df.iloc[test_idx].copy()

baseline_test["baseline_score"] = (
    0.40 * baseline_test["impressions_90d"].rank(pct=True)
    + 0.35 * baseline_test["avg_position"].rank(pct=True)
    + 0.25 * (-baseline_test["impression_change_pct"]).rank(pct=True)
)

baseline_p50 = precision_at_k(
    baseline_test["review_label"].reset_index(drop=True),
    baseline_test["baseline_score"].values,
    50
)

comparison = pd.DataFrame({
    "Method": [
        "Baseline",
        "Random Forest"
    ],
    "Precision@50": [
        baseline_p50,
        model_p50
    ]
})

display(comparison)

,Method,Precision@50
0,Baseline,1.00
1,Random Forest,0.72


The comparison should be interpreted cautiously because the provisional proxy is constructed from observed signals that are also available to the model. Therefore, a higher model score would not by itself prove that the model produces better real-world recommendations. A later validation step is needed to test whether the ranking generalizes beyond the proxy definition.

## 4. Errors and interpretation

I will inspect the highest-scored model pages and compare them with the proxy label. False positives are pages ranked highly by the model that do not meet the provisional proxy threshold, while false negatives are pages with the proxy label that the model ranks lower. The model's feature importances will also be inspected to understand which signals it relies on. These results are directional and should not be interpreted as causal explanations.

In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Model predictions
test_results = work_df.iloc[test_idx].copy()

test_results["model_score"] = model_scores
test_results["actual_label"] = y_test.values

test_results["predicted_top50"] = False

top50_indices = np.argsort(model_scores)[::-1][:50]
test_results.iloc[top50_indices,
                  test_results.columns.get_loc("predicted_top50")] = True

print("=== Top 10 model-ranked pages ===")

display(
    test_results
    .sort_values("model_score", ascending=False)
    [
        [
            "content_id",
            "model_score",
            "actual_label",
            "impressions_90d",
            "avg_position",
            "impression_change_pct"
        ]
    ]
    .head(10)
)

print("\n=== Feature importance ===")

importance = pd.DataFrame({
    "feature": feature_columns,
    "importance": model.feature_importances_
}).sort_values(
    "importance",
    ascending=False
)

display(importance.head(10))

=== Top 10 model-ranked pages ===


,content_id,model_score,actual_label,impressions_90d,avg_position,impression_change_pct
4076,content_66458ac1b739,0.700932,0,6822,2.9,-0.787661
11676,content_dbf29df094f9,0.700717,1,4519,44.7,-0.399889
22049,content_da33a53074bd,0.694882,0,2678,5.7,-0.659658
6228,content_e988c1699454,0.649523,1,2197,21.5,-0.436828
16876,content_1f97c2c8f517,0.647676,1,2156,35.7,-0.289474
13572,content_ccf887ee3581,0.636593,0,2993,5.5,-0.708642
29392,content_8a1fbaf0f871,0.635638,1,4963,8.6,-0.445575
3245,content_03d074e8f486,0.633346,1,6582,6.9,-0.615196
13213,content_f2508318df4b,0.631589,0,2831,3.1,-0.788360
10136,content_df71843dcd17,0.629107,1,27334,76.4,0.441496



=== Feature importance ===


,feature,importance
18,impressions_prev_30d,0.211906
5,impressions_90d,0.135416
3,word_count,0.104299
4,char_count,0.089549
13,days_with_impressions,0.069319
15,impressions_last_30d,0.065464
21,content_age_days,0.046503
7,pageviews_90d,0.039890
12,scroll_events_90d,0.027971
16,clicks_last_30d,0.027804


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.